# Step 1 — Disfluency BIO training (NusaBERT-base)

Fine-tunes `LazarusNLP/NusaBERT-base` (`AutoModelForTokenClassification`) on `data/disfluency/{train,val,test}.jsonl` (`3.1_disfluency_bio_labeling.ipynb`'s output) over the 7-tag scheme (`O`, `B-IP`, `B-FS`, `B-RC`, `I-RC`, `B-RP`, `B-RM`). Mirrors `4.2_ner_training.ipynb`'s structure (`WeightedTrainer`, seqeval `compute_metrics`), since this dataset has even sharper imbalance — `B-IP`=818 vs `B-FS`/`B-RP`/`B-RM` in single digits.

In [1]:
import json
from pathlib import Path

import torch
import numpy as np

SEED = 42
MAX_LENGTH = 64
MODEL_NAME = "indobenchmark/indobert-base-p2"


In [2]:
label_map = json.loads(Path("../data/disfluency/label_map.json").read_text())
label2id = {k: int(v) for k, v in label_map.items()}
id2label = {v: k for k, v in label2id.items()}
LABEL_LIST = [id2label[i] for i in range(len(id2label))]
print(LABEL_LIST)

['B-FS', 'B-IP', 'B-RC', 'B-RM', 'B-RP', 'I-RC', 'O']


## Step 2 — Load splits

In [3]:
from datasets import Dataset

def load_split(jsonl_path: str) -> Dataset:
    records = []
    with open(jsonl_path, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            records.append({
                "input_ids": rec["input_ids"],
                "attention_mask": rec["attention_mask"],
                "labels": rec["labels"],
            })
    return Dataset.from_list(records)


train_dataset = load_split("../data/disfluency/train.jsonl")
val_dataset   = load_split("../data/disfluency/val.jsonl")
test_dataset  = load_split("../data/disfluency/test.jsonl")

print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1648
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 206
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 206
})


## Step 3 — Class imbalance

Tags skew heavily toward `O` (~93% of tokens) with `B-FS`/`B-RP`/`B-RM` near-absent in train (forced there entirely by `3.1`'s split logic since they're too rare to stratify). Inverse-frequency class weights computed from the **train split only**, same convention as `3.2_bio_labeling.ipynb`/`4.2_ner_training.ipynb`.

In [4]:
from collections import Counter

def compute_label_counts(jsonl_path: str) -> Counter:
    """Count label ids across a split, excluding -100 (ignored/padding)."""
    counts = Counter()
    with open(jsonl_path, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            for lbl_id in rec["labels"]:
                if lbl_id != -100:
                    counts[lbl_id] += 1
    return counts


train_label_counts = compute_label_counts("../data/disfluency/train.jsonl")
for lbl_id in range(len(LABEL_LIST)):
    print(f"  {id2label[lbl_id]:8s}: {train_label_counts[lbl_id]:6d}")

  B-FS    :     26
  B-IP    :    680
  B-RC    :     30
  B-RM    :     26
  B-RP    :     26
  I-RC    :     30
  O       :  15651


In [5]:
total_tokens = sum(train_label_counts.values())
n_classes = len(LABEL_LIST)

class_weights = torch.tensor(
    [total_tokens / (n_classes * max(train_label_counts[i], 1)) for i in range(n_classes)],
    dtype=torch.float,
)

for lbl_id, weight in enumerate(class_weights.tolist()):
    print(f"  {id2label[lbl_id]:8s}: weight={weight:.3f}")

loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)

  B-FS    : weight=90.489
  B-IP    : weight=3.460
  B-RC    : weight=78.424
  B-RM    : weight=90.489
  B-RP    : weight=90.489
  I-RC    : weight=78.424
  O       : weight=0.150


### Sanity check

In [6]:
assert class_weights.argmin().item() == label2id["O"], "O should get the lowest weight"
assert loss_fn.ignore_index == -100
print("Step 3 sanity checks passed.")

Step 3 sanity checks passed.


## Step 4 — Model architecture

Full fine-tune of `LazarusNLP/NusaBERT-base` via `AutoModelForTokenClassification`.

In [7]:
from transformers import AutoModelForTokenClassification

device = (
    torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available()
    else torch.device("cpu")
)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id,
)
model.to(device)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"device          : {device}")
print(f"total params    : {n_params:,}")
print(f"trainable params: {n_trainable:,}  (full fine-tune — should equal total)")

[transformers] You passed `num_labels=7` which is incompatible to the `id2label` map of length `5`.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


device          : mps
total params    : 123,856,135
trainable params: 123,856,135  (full fine-tune — should equal total)


### Sanity check: forward pass on a real batch

In [8]:
sample_batch = train_dataset[:4]
batch_input_ids = torch.tensor(sample_batch["input_ids"], device=device)
batch_attention_mask = torch.tensor(sample_batch["attention_mask"], device=device)
batch_labels = torch.tensor(sample_batch["labels"], device=device)

model.eval()
with torch.no_grad():
    outputs = model(input_ids=batch_input_ids, attention_mask=batch_attention_mask)

logits = outputs.logits
print(f"logits shape: {tuple(logits.shape)}  (expect (4, {MAX_LENGTH}, {len(LABEL_LIST)}))")
assert logits.shape == (4, MAX_LENGTH, len(LABEL_LIST))

loss_fn = loss_fn.to(device)
sample_loss = loss_fn(logits.view(-1, len(LABEL_LIST)), batch_labels.view(-1))
print(f"sample weighted loss: {sample_loss.item():.4f}")

model.train()
print("Step 4 sanity checks passed.")

logits shape: (4, 64, 7)  (expect (4, 64, 7))
sample weighted loss: 2.4244
Step 4 sanity checks passed.


## Step 5 — Training

`WeightedTrainer` overrides `compute_loss` to use Step 3's class-weighted loss instead of `Trainer`'s unweighted default.

In [9]:
import evaluate

seqeval_metric = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    """
    Convert per-token logits/labels back into BIO tag sequences (dropping
    -100 positions) and score with seqeval's entity-level precision/recall/F1.
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    true_seqs, pred_seqs = [], []
    for pred_row, label_row in zip(preds, labels):
        true_seq = [id2label[l] for l in label_row if l != -100]
        pred_seq = [id2label[p] for p, l in zip(pred_row, label_row) if l != -100]
        true_seqs.append(true_seq)
        pred_seqs.append(pred_seq)

    results = seqeval_metric.compute(predictions=pred_seqs, references=true_seqs)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [10]:
from transformers import Trainer, TrainingArguments

class WeightedTrainer(Trainer):
    """Trainer with class-weighted CrossEntropyLoss (Step 3) instead of the unweighted default."""

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        weighted_loss_fn = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device), ignore_index=-100
        )
        loss = weighted_loss_fn(logits.view(-1, len(LABEL_LIST)), labels.view(-1))

        return (loss, outputs) if return_outputs else loss


training_args = TrainingArguments(
    output_dir="../models/nusabert-disfluency-bio",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_strategy="epoch",
    report_to="none",
    seed=SEED,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [11]:
train_result = trainer.train()
train_result.metrics

/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.788968,0.008577,0.844828,1.000000,0.915888,0.990957
2,0.037188,0.000490,0.989899,1.000000,0.994924,0.999524
3,0.067312,0.000142,1.000000,1.000000,1.000000,1.000000
4,0.045476,0.000128,1.000000,1.000000,1.000000,1.000000
5,0.038028,0.000087,1.000000,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': 228.6762,
 'train_samples_per_second': 36.033,
 'train_steps_per_second': 2.252,
 'total_flos': 269147825633280.0,
 'train_loss': 0.1953942965535284,
 'epoch': 5.0}

### Save the fine-tuned model

In [12]:
from transformers import AutoTokenizer

final_model_path = "../models/nusabert-disfluency-bio-final"
trainer.save_model(final_model_path)
AutoTokenizer.from_pretrained(MODEL_NAME).save_pretrained(final_model_path)
print(f"Saved fine-tuned model -> {final_model_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Saved fine-tuned model -> ../models/nusabert-disfluency-bio-final


## Step 6 — Evaluation

Run on the held-out **test** split — not seen during training or `load_best_model_at_end` checkpoint selection — and report per-tag-type precision/recall/F1, not just overall, since rare tags (`FS`/`RP`/`RM`) are where this model is most likely to be weak (or untestable if `3.1`'s split forced all their occurrences into train).

In [13]:
test_predictions = trainer.predict(test_dataset)
test_logits, test_label_ids = test_predictions.predictions, test_predictions.label_ids
test_preds = np.argmax(test_logits, axis=-1)

print("overall test metrics:")
for k, v in test_predictions.metrics.items():
    print(f"  {k:25s}: {v}")

/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


overall test metrics:
  test_loss                : 0.23493525385856628
  test_precision           : 0.9795918367346939
  test_recall              : 0.9896907216494846
  test_f1                  : 0.9846153846153847
  test_accuracy            : 0.9990191270230505
  test_runtime             : 0.8675
  test_samples_per_second  : 237.468
  test_steps_per_second    : 14.986


In [14]:
test_true_seqs, test_pred_seqs = [], []
for pred_row, label_row in zip(test_preds, test_label_ids):
    true_seq = [id2label[l] for l in label_row if l != -100]
    pred_seq = [id2label[p] for p, l in zip(pred_row, label_row) if l != -100]
    test_true_seqs.append(true_seq)
    test_pred_seqs.append(pred_seq)

per_tag_report = seqeval_metric.compute(
    predictions=test_pred_seqs, references=test_true_seqs
)

print("Per-tag-type breakdown (test set):")
for tag_type, scores in per_tag_report.items():
    if tag_type.startswith("overall"):
        continue
    print(
        f"  {tag_type:10s}  precision={scores['precision']:.3f}  "
        f"recall={scores['recall']:.3f}  f1={scores['f1']:.3f}  "
        f"n={scores['number']}"
    )
print()
print(f"  overall precision={per_tag_report['overall_precision']:.3f}  "
      f"recall={per_tag_report['overall_recall']:.3f}  "
      f"f1={per_tag_report['overall_f1']:.3f}  "
      f"accuracy={per_tag_report['overall_accuracy']:.3f}")

Per-tag-type breakdown (test set):
  FS          precision=1.000  recall=1.000  f1=1.000  n=4
  IP          precision=1.000  recall=1.000  f1=1.000  n=83
  RC          precision=0.600  recall=0.750  f1=0.667  n=4
  RM          precision=1.000  recall=1.000  f1=1.000  n=3
  RP          precision=1.000  recall=1.000  f1=1.000  n=3

  overall precision=0.980  recall=0.990  f1=0.985  accuracy=0.999


## Step 7 — Inference + repair

Loads the saved model fresh from disk to confirm standalone reload. `predict_disfluency(text)` tokenizes raw text, decodes subword predictions back to word-level BIO tags (first-subword convention), and extracts contiguous `B-X`/`I-X` spans. `repair(text)` applies the project's repair strategy: delete `IP`/`RP`/`FS` spans entirely (filler, abandoned/wrong value, truncated word — none belong in the cleaned transcript); `RM` (the correct value) is left in place; `RC` repeats drop the first occurrence and keep the second as the fluent form.

In [15]:
from transformers import AutoModelForTokenClassification as _AMFTC, AutoTokenizer as _AT

INFERENCE_MODEL_PATH = "../models/nusabert-disfluency-bio-final"

inference_tokenizer = _AT.from_pretrained(INFERENCE_MODEL_PATH)
inference_model = _AMFTC.from_pretrained(INFERENCE_MODEL_PATH)
inference_model.to(device)
inference_model.eval()

print(f"Loaded model from {INFERENCE_MODEL_PATH}")
print(f"id2label: {inference_model.config.id2label}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded model from ../models/nusabert-disfluency-bio-final
id2label: {0: 'B-FS', 1: 'B-IP', 2: 'B-RC', 3: 'B-RM', 4: 'B-RP', 5: 'I-RC', 6: 'O'}


In [16]:
DELETE_TAGS = {"IP", "RP", "FS"}


def predict_disfluency(text: str) -> dict:
    """
    Run raw text through the fine-tuned disfluency tagger.

    Returns:
      tokens : word-level tokens (text.split())
      labels : predicted BIO tag per word (first-subword label only)
      spans  : list of {label: <TAG>, text: str, start: int, end: int}
               (end is exclusive, word indices)
    """
    tokens = text.split()
    if not tokens:
        return {"tokens": [], "labels": [], "spans": []}

    encoding = inference_tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    word_ids = encoding.word_ids()
    encoding = {k: v.to(device) for k, v in encoding.items()}

    with torch.no_grad():
        logits = inference_model(**encoding).logits[0]
    pred_ids = logits.argmax(dim=-1).tolist()

    word_labels = [None] * len(tokens)
    prev_word_id = None
    for tok_idx, word_id in enumerate(word_ids):
        if word_id is not None and word_id != prev_word_id:
            word_labels[word_id] = id2label[pred_ids[tok_idx]]
        prev_word_id = word_id
    word_labels = [lbl if lbl is not None else "O" for lbl in word_labels]

    spans = []
    i = 0
    while i < len(tokens):
        lbl = word_labels[i]
        if lbl.startswith("B-"):
            span_type = lbl[2:]
            j = i + 1
            while j < len(tokens) and word_labels[j] == f"I-{span_type}":
                j += 1
            spans.append({"label": span_type, "text": " ".join(tokens[i:j]), "start": i, "end": j})
            i = j
        else:
            i += 1

    return {"tokens": tokens, "labels": word_labels, "spans": spans}


def repair(text: str) -> str:
    """
    Delete IP (filler), RP (reparandum -- the wrong value being corrected), and
    FS (truncated false-start word) spans outright. RM (repair -- the correct
    value) is left untouched since it's what should remain in the cleaned
    transcript. RC (repeat) tags both duplicate tokens via B-/I-, so drop the
    first occurrence and keep the second as the fluent surface form.
    """
    result = predict_disfluency(text)
    tokens, spans = result["tokens"], result["spans"]
    drop_indices = set()
    for span in spans:
        if span["label"] in DELETE_TAGS:
            drop_indices.update(range(span["start"], span["end"]))
        elif span["label"] == "RC":
            drop_indices.update(range(span["start"], span["end"] - 1))
    kept = [tok for idx, tok in enumerate(tokens) if idx not in drop_indices]
    return " ".join(kept)

### Try it on fresh, unseen sentences

In [17]:
demo_sentences = [
    "eh, gue mau pesen nasi goreng spesial dua porsi",
    "mas, gue mau, mau pesen ayam goreng satu porsi",
    "eh gue pesen mie goreng spesial dua, eh, tiga porsi pak",
    "na- nasi goreng ikan asin satu ya kak",
    "anu bang, tambah ayam bakar satu lagi ya, buat semua",
]

for text in demo_sentences:
    result = predict_disfluency(text)
    print(f"input  : {text}")
    print(f"tagged : {' '.join(f'{t}[{l}]' if l != 'O' else t for t, l in zip(result['tokens'], result['labels']))}")
    print(f"spans  : {result['spans']}")
    print(f"repaired: {repair(text)}")
    print()

input  : eh, gue mau pesen nasi goreng spesial dua porsi
tagged : eh,[B-IP] gue mau pesen nasi goreng spesial dua porsi
spans  : [{'label': 'IP', 'text': 'eh,', 'start': 0, 'end': 1}]
repaired: gue mau pesen nasi goreng spesial dua porsi

input  : mas, gue mau, mau pesen ayam goreng satu porsi
tagged : mas, gue mau,[B-RC] mau[I-RC] pesen ayam goreng satu porsi
spans  : [{'label': 'RC', 'text': 'mau, mau', 'start': 2, 'end': 4}]
repaired: mas, gue mau pesen ayam goreng satu porsi

input  : eh gue pesen mie goreng spesial dua, eh, tiga porsi pak
tagged : eh[B-IP] gue pesen mie goreng spesial dua,[B-RP] eh,[B-IP] tiga[B-RM] porsi pak
spans  : [{'label': 'IP', 'text': 'eh', 'start': 0, 'end': 1}, {'label': 'RP', 'text': 'dua,', 'start': 6, 'end': 7}, {'label': 'IP', 'text': 'eh,', 'start': 7, 'end': 8}, {'label': 'RM', 'text': 'tiga', 'start': 8, 'end': 9}]
repaired: gue pesen mie goreng spesial tiga porsi pak

input  : na- nasi goreng ikan asin satu ya kak
tagged : na-[B-FS] nasi goreng i